In [1]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

/var/folders/qs/knmt2c551rv0l8dbzw06zgsc0000gn/T/ipykernel_20725/3777615979.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython.display
  from IPython.core.display import display, HTML


# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [4]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("../data/kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


### Let's divide the training and test set into two partitions

In [ ]:
# Your code
#does not make sence cause data foulder already contains train and test data. Better to shorten test data also to speed up the development.

data_test = pd.read_csv("../data/kg_test.csv",encoding='latin-1')
data_test = data_test.head(1000)
data_test.fillna("",inplace=True)


(1000, 1)


## Data Preprocessing

In [15]:
import string
import nltk
nltk.download('stopwords')
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/katebolshakova/nltk_data...


!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


[nltk_data]   Unzipping corpora/stopwords.zip.


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [22]:
data.head(3)
data.iloc[0, 0]

"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL I AM MIKE CHUKWU , THE MANAGER, BILLS AND EXCHANGE AT THE FOREIGN REMITTANCE DEPARTMENT OF THE ZENITH INTERNATIONAL BANK PLC. I AM WRITING THIS LETTER TO ASK FOR YOUR SUPPORT AND COOPERATION TO CARRY OUT THIS BUSINESS OPPORTUNITY IN MY DEPARTMENT. WE DISCOVERED AN ABANDONED SUM OF $15,000,000.00 (FIFTEEN MILLION UNITED STATES DOLLARS ONLY) IN AN ACCOUNT THAT BELONGS TO ONE OF OUR FOREIGN CUSTOMERS WHO DIED ALONG WITH HIS ENTIRE FAMILY OF A WIFE AND TWO CHILDREN IN NOVEMBER 1997 IN A PLANE CRASH. SINCE WE HEARD OF HIS DEATH, WE HAVE BEEN EXPECTING HIS NEXT-OF-KIN TO COME OVER AND PUT CLAIMS FOR HIS MONEY AS THE HEIR,BECAUSE WE CANNOT RELEASE THE FUND FROM HIS ACCOUNT UNLESS SOMEONE APPLIES FOR CLAIM AS THE NEXT-OF-KIN TO THE DECEASED AS INDICATED IN OUR BANKING GUIDELINES. UNFORTUNATELY, NEITHER THEIR FAMILY MEMBER NOR DISTANT RELATIVE HAS EVER APPEARED TO CLAIM THE SAID FUND. UPON THIS DISCOVERY,I AND OTHER OFFICIALS IN MY DEPARTMENT HAVE

In [23]:
# Using regex
import re

def preprocess_text(text):
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    # Remove single characters
    text = re.sub(r'\s+[a-zA-Z]\s+', ' ', text)
    # Remove single character at start
    text = re.sub(r'^[a-zA-Z]\s+', '', text)
    # Remove multiple space
    text = re.sub(r'\s+', ' ', text)
    # Remove prefixed b
    text = re.sub(r'^b\s+', '', text)
    # Lowercase
    text = text.lower()

    return text

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [ ]:
# Your code

## Now let's work on removing stopwords
Remove the stopwords.

In [24]:
# Your code
def preprocess_stopwords(text):
    stop_words = set(stopwords.words('english'))
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    return ' '.join(filtered_words)

In [26]:
#Put it all together
data.iloc[:,0] = data.iloc[:,0].apply(preprocess_text)
data_test.iloc[:,0] = data_test.iloc[:,0].apply(preprocess_text)

data.iloc[:,0] = data.iloc[:,0].apply(preprocess_stopwords)
data_test.iloc[:,0] = data_test.iloc[:,0].apply(preprocess_stopwords)

## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [27]:
# Your code
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()



[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/katebolshakova/nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/katebolshakova/nltk_data...


In [33]:
data.iloc[:,0] = data.iloc[:,0].apply(lambda x: ' '.join([lemmatizer.lemmatize(word) for word in x.split()]))
data_test.iloc[:,0] = data_test.iloc[:,0].apply(lambda x: ' '.join([lemmatizer.lemmatize(word) for word in x.split()]))

data.iloc[0, 0]


'dear sir strictly private business proposal mike chukwu manager bill exchange foreign remittance department zenith international bank plc writing letter ask support cooperation carry business opportunity department discovered abandoned sum fifteen million united state dollar account belongs one foreign customer died along entire family wife two child november plane crash since heard death expecting next kin come put claim money heir cannot release fund account unless someone applies claim next kin deceased indicated banking guideline unfortunately neither family member distant relative ever appeared claim said fund upon discovery official department agreed make business release total amount account heir fund since one came discovered maintained account bank otherwise fund returned bank treasury unclaimed fund agreed ratio sharing stated thus foreign partner u official department settlement local foreign expences incurred u course business upon successful completion transfer one collea

## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [40]:
# Your code
from collections import Counter

ham = data[data['label'] == '0']
spam = data[data['label'] == '1']

ham_text = ' '.join(ham.iloc[:,0])
spam_text = ' '.join(spam.iloc[:,0])

spam_counts = Counter(spam_text.split())
ham_counts = Counter(ham_text.split())

spam_common = spam_counts.most_common(10)
ham_common = ham_counts.most_common(10)

print(spam_common)
print(ham_common)

data.head()

[]
[]


,text,label
0,dear sir strictly private business proposal mi...,1
1,,0
2,nora cheryl emailed dozen memo haiti weekend p...,0
3,dear sir fmadam know proposal might surprise e...,1
4,fyi,0


## Extra features

In [41]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data['money_mark'] = data['text'].str.contains(money_simbol_list)*1
data['suspicious_words'] = data['text'].str.contains(suspicious_words)*1
data['text_len'] = data['text'].apply(len)

data_test['money_mark'] = data_test['text'].str.contains(money_simbol_list)*1
data_test['suspicious_words'] = data_test['text'].str.contains(suspicious_words)*1
data_test['text_len'] = data_test['text'].apply(len)

data.head()

,text,label,money_mark,suspicious_words,text_len
0,dear sir strictly private business proposal mi...,1,1,1,1493
1,,0,0,0,0
2,nora cheryl emailed dozen memo haiti weekend p...,0,0,0,111
3,dear sir fmadam know proposal might surprise e...,1,1,1,1346
4,fyi,0,0,0,3


## How would you create a Bag of Words with the CountVectorizer method?

In [ ]:
# Your code

from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()

X_train = vectorizer.fit_transform(data['text'])
X_test = vectorizer.transform(data_test['text'])
X_test.shape


(1000, 29875)

## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [47]:
# Your code

from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(data['text'])
X_test_tfidf = tfidf_vectorizer.transform(data_test['text'])
X_test_tfidf.shape

(1000, 29875)

### Extra Task (optional) - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

Use a MultinimialNB with default parameters.

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [59]:
# Your code

from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report


from scipy.sparse import hstack

X_train_part, X_val, y_train_part, y_val = train_test_split(

    X_train_tfidf,

    data['label'],

    test_size=0.2,

    random_state=42

)

model_tfidf = MultinomialNB()

model_tfidf.fit(X_train_part, y_train_part)

pred_val = model_tfidf.predict(X_val)

print("Accuracy:", accuracy_score(y_val, pred_val))

print(classification_report(y_val, pred_val))


Accuracy: 0.94
              precision    recall  f1-score   support

           0       1.00      0.90      0.95       125
           1       0.86      1.00      0.93        75

    accuracy                           0.94       200
   macro avg       0.93      0.95      0.94       200
weighted avg       0.95      0.94      0.94       200



In [ ]:
# BoW accuracy 0.96
#Bow with extra features accuracy 0.9
# TF-IDF accuracy 0.94

Index(['text', 'money_mark', 'suspicious_words', 'text_len'], dtype='object')